# 11 - Compute Automatic Eval

### Setup

In [1]:
!pip install -q --upgrade "bert-score==0.3.13" "transformers==4.46.3" "tokenizers==0.20.3" "huggingface-hub==0.26.2" "rouge-score==0.1.2"
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 29.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.26.2 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.26.2 which is incompatible.


In [2]:
import os
import re
import string
import collections
import pandas as pd
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score
from huggingface_hub import HfApi

## 1. Konfigurasi

In [3]:
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_Faiss")
    except ImportError:
        token = os.environ.get("HF_Faiss")
        if token is None:
            raise RuntimeError("HF_TOKEN tidak ditemukan.")
        return token
 
HF_TOKEN = get_hf_token()
HF_REPO = "Makaareeem/publikasi-rag-finetuning-dataset"
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else "/workspace")
 
_api_check = HfApi(token=HF_TOKEN)
if not _api_check.repo_exists(HF_REPO, repo_type="dataset"):
    _api_check.create_repo(HF_REPO, repo_type="dataset", private=True)
    print(f"Dataset repo dibuat: {HF_REPO}")
 
INDOBERT_MODEL = "firqaaa/indo-sentence-bert-base"
BERTSCORE_BATCH_SIZE = 16
 
import torch
_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"IndoBERTScore device: {_DEVICE}")
 
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

IndoBERTScore device: cuda


In [4]:
def normalize_answer(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = " ".join(text.split())
    return text

## 2. Fungsi Metric

### Exact matching

In [5]:
# Ekstrak semua angka dari teks (mendukung format Indonesia: koma sbg desimal, titik sbg pemisah ribuan)
def extract_numbers(text):
    if pd.isna(text):
        return set()
    text = str(text)
    raw_numbers = re.findall(r"\d{1,3}(?:\.\d{3})*(?:,\d+)?|\d+(?:,\d+)?", text)
    normalized = set()
    for n in raw_numbers:
        n_clean = n.replace(".", "").replace(",", ".")
        try:
            normalized.add(float(n_clean))
        except ValueError:
            continue
    return normalized

In [6]:
def compute_em(pred, ref):
    ref_numbers = extract_numbers(ref)
    if len(ref_numbers) == 0:
        return None
    pred_numbers = extract_numbers(pred)
    return int(ref_numbers.issubset(pred_numbers))

### F1-score

In [7]:
def compute_f1(pred, ref):
    pred_tokens = normalize_answer(pred).split()
    ref_tokens = normalize_answer(ref).split()
    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return int(pred_tokens == ref_tokens)
    common = collections.Counter(pred_tokens) & collections.Counter(ref_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)
    return (2 * precision * recall) / (precision + recall)

### Rouge-l

In [8]:
def compute_rouge_l(pred, ref):
    pred, ref = normalize_answer(pred), normalize_answer(ref)
    if not pred or not ref:
        return 0.0
    return _rouge.score(ref, pred)["rougeL"].fmeasure

### Indobert score

In [9]:
from transformers import AutoTokenizer
_indobert_tokenizer = AutoTokenizer.from_pretrained(INDOBERT_MODEL)
 
def compute_indobertscore(preds, refs, batch_size=16, max_tokens=510):
    def _truncate(text):
        if not isinstance(text, str) or not text.strip():
            return "[KOSONG]"
        ids = _indobert_tokenizer.encode(text, add_special_tokens=False, truncation=True, max_length=max_tokens)
        return _indobert_tokenizer.decode(ids, skip_special_tokens=True)
 
    preds = [_truncate(p) for p in preds]
    refs = [_truncate(r) for r in refs]
 
    all_f1 = []
    for i in range(0, len(preds), batch_size):
        batch_preds = preds[i:i + batch_size]
        batch_refs = refs[i:i + batch_size]
        _, _, F1 = bertscore_score(
            batch_preds, batch_refs,
            model_type=INDOBERT_MODEL,
            num_layers=12,
            verbose=False,
            batch_size=batch_size,
            device=_DEVICE,
        )
        all_f1.extend(F1.tolist())
        print(f"indobertscore {i + len(batch_preds)}/{len(preds)}")
    return all_f1

tokenizer_config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

### Cek kolom eval

In [10]:
REQUIRED_COLUMNS = ["ans_ref", "ans_pred_base", "ans_pred_finetuned"]

In [11]:
def load_and_validate(csv_path):
    df = pd.read_csv(csv_path, storage_options={"token": HF_TOKEN} if csv_path.startswith("hf://") else None)
 
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Kolom hilang di {csv_path}: {missing}")
 
    n_before = len(df)
    for col in REQUIRED_COLUMNS:
        n_nan = df[col].isna().sum()
        if n_nan > 0:
            print(f"[WARNING] {csv_path} — kolom '{col}' punya {n_nan} NaN, diisi string kosong")
            df[col] = df[col].fillna("")
 
    print(f"Loaded {csv_path}: {n_before} baris")
    return df

## 3. Main Program

In [12]:
def evaluate_condition(df, pred_column, ref_column="ans_ref"):
    df = df.copy()
    df["em"] = df.apply(lambda row: compute_em(row[pred_column], row[ref_column]), axis=1)
    df["f1"] = df.apply(lambda row: compute_f1(row[pred_column], row[ref_column]), axis=1)
    df["rouge_l"] = df.apply(lambda row: compute_rouge_l(row[pred_column], row[ref_column]), axis=1)
    df["indobertscore"] = compute_indobertscore(df[pred_column].tolist(), df[ref_column].tolist())
 
    print(f"EM             : {df['em'].mean():.4f}")
    print(f"F1             : {df['f1'].mean():.4f}")
    print(f"ROUGE-L        : {df['rouge_l'].mean():.4f}")
    print(f"IndoBERTScore  : {df['indobertscore'].mean():.4f}")
    return df

### Load prediksi

In [13]:
df_gemma = load_and_validate(f"hf://datasets/{HF_REPO}/eval_gemma2_predictions_gguf.csv")
df_llama = load_and_validate(f"hf://datasets/{HF_REPO}/eval_llama3.2_predictions_gguf.csv")

Loaded hf://datasets/Makaareeem/publikasi-rag-finetuning-dataset/eval_gemma2_predictions_gguf.csv: 1791 baris
Loaded hf://datasets/Makaareeem/publikasi-rag-finetuning-dataset/eval_llama3.2_predictions_gguf.csv: 1791 baris


In [14]:
## GEMMA-2 - BASE MODEL (SEBELUM FINE-TUNING)
 
df_gemma_base = evaluate_condition(df_gemma, pred_column="ans_pred_base")
df_gemma_base

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

indobertscore 16/1791
indobertscore 32/1791
indobertscore 48/1791
indobertscore 64/1791
indobertscore 80/1791
indobertscore 96/1791
indobertscore 112/1791
indobertscore 128/1791
indobertscore 144/1791
indobertscore 160/1791
indobertscore 176/1791
indobertscore 192/1791
indobertscore 208/1791
indobertscore 224/1791
indobertscore 240/1791
indobertscore 256/1791
indobertscore 272/1791
indobertscore 288/1791
indobertscore 304/1791
indobertscore 320/1791
indobertscore 336/1791
indobertscore 352/1791
indobertscore 368/1791
indobertscore 384/1791
indobertscore 400/1791
indobertscore 416/1791
indobertscore 432/1791
indobertscore 448/1791
indobertscore 464/1791
indobertscore 480/1791
indobertscore 496/1791
indobertscore 512/1791
indobertscore 528/1791
indobertscore 544/1791
indobertscore 560/1791
indobertscore 576/1791
indobertscore 592/1791
indobertscore 608/1791
indobertscore 624/1791
indobertscore 640/1791
indobertscore 656/1791
indobertscore 672/1791
indobertscore 688/1791
indobertscore 704

,record_id,instruction,ans_ref,ans_pred_base,ans_pred_finetuned,em,f1,rouge_l,indobertscore
0,b05d48cbd7388514,Apa rekomendasi pertama dari KPK yang disusun ...,"Teks menyatakan ""Pertama, diperlukan pengemban...","Berdasarkan konteks yang Anda berikan, saya ti...",Teks menyatakan bahwa rekomendasi pertama adal...,NaN,0.216216,0.216216,0.511699
1,6fd9fbd0051e170c,Siapa yang memprakarsai GONE theory dan apa sa...,"Teks menyatakan ""GONE theory yang diprakarsai ...",GONE theory yang diprakarsai oleh Jack Bologne...,Teks menyatakan GONE theory diprakarsai oleh J...,NaN,0.554217,0.530120,0.821402
2,9b46bc8bdc31522e,Menurut pendekatan lingkungan maupun kultural ...,"Teks menyatakan ""Dalam pendekatan lingkungan m...",Dalam pendekatan lingkungan maupun kultural ma...,Teks menyatakan bahwa solusi dalam pendekatan ...,NaN,0.360000,0.333333,0.699521
3,7e0baf29d0fa19e6,Sebutkan empat langkah pencegahan khusus korup...,"Teks menyatakan ""Heyneman (2004) menyatakan pa...","Berdasarkan konteks yang diberikan, dan inform...",Teks menyatakan bahwa Heyneman (2004) menyajik...,1.0,0.315789,0.189474,0.554579
4,8eaf6490b590c374,Apa yang dimaksud dengan GONE theory dalam kon...,"Teks menyatakan ""GONE theory yang diprakarsai ...","GONE theory, yang diprakarsai oleh Jack Bologn...",Teks menyatakan GONE theory menjelaskan empat ...,NaN,0.396135,0.280193,0.709542
...,...,...,...,...,...,...,...,...,...
1786,03460d992d3fd3a0,Apa sumber data yang digunakan untuk menghasil...,"Teks menyatakan ""Badan Pusat Statistik, Survei...","Berdasarkan konteks yang Anda berikan, sumber ...",Teks menyatakan sumber data adalah Badan Pusat...,NaN,0.457831,0.409639,0.717005
1787,3f25cc3863855ead,Apa yang diukur oleh indikator 'Persentase Pen...,"Teks menyatakan indikator tersebut adalah ""Per...",Indikator 'Persentase Penduduk yang Pernah Raw...,Teks menyatakan bahwa tabel ini mengukur perse...,NaN,0.512195,0.463415,0.862529
1788,94755286a0fca43d,Di bagian mana data mengenai akses rumah tangg...,"Teks menyatakan ""Lanjutan Tabel / Continued Ta...",Data mengenai akses rumah tangga terhadap fasi...,"Teks menyatakan ""Lanjutan Tabel / Continued Ta...",1.0,0.452830,0.452830,0.811166
1789,c97c2a058036c218,Di bagian mana informasi tentang sumber data u...,"Teks menyatakan ""Sumber/ Source : Badan Pusat ...",Informasi tentang sumber data untuk tabel samp...,Teks menyatakan bahwa sumber data untuk tabel ...,NaN,0.203390,0.169492,0.498026


In [15]:
## GEMMA-2 - FINE-TUNED MODEL (SESUDAH FINE-TUNING)
 
df_gemma_ft = evaluate_condition(df_gemma, pred_column="ans_pred_finetuned")
df_gemma_ft

indobertscore 16/1791
indobertscore 32/1791
indobertscore 48/1791
indobertscore 64/1791
indobertscore 80/1791
indobertscore 96/1791
indobertscore 112/1791
indobertscore 128/1791
indobertscore 144/1791
indobertscore 160/1791
indobertscore 176/1791
indobertscore 192/1791
indobertscore 208/1791
indobertscore 224/1791
indobertscore 240/1791
indobertscore 256/1791
indobertscore 272/1791
indobertscore 288/1791
indobertscore 304/1791
indobertscore 320/1791
indobertscore 336/1791
indobertscore 352/1791
indobertscore 368/1791
indobertscore 384/1791
indobertscore 400/1791
indobertscore 416/1791
indobertscore 432/1791
indobertscore 448/1791
indobertscore 464/1791
indobertscore 480/1791
indobertscore 496/1791
indobertscore 512/1791
indobertscore 528/1791
indobertscore 544/1791
indobertscore 560/1791
indobertscore 576/1791
indobertscore 592/1791
indobertscore 608/1791
indobertscore 624/1791
indobertscore 640/1791
indobertscore 656/1791
indobertscore 672/1791
indobertscore 688/1791
indobertscore 704

,record_id,instruction,ans_ref,ans_pred_base,ans_pred_finetuned,em,f1,rouge_l,indobertscore
0,b05d48cbd7388514,Apa rekomendasi pertama dari KPK yang disusun ...,"Teks menyatakan ""Pertama, diperlukan pengemban...","Berdasarkan konteks yang Anda berikan, saya ti...",Teks menyatakan bahwa rekomendasi pertama adal...,NaN,0.853333,0.853333,0.934062
1,6fd9fbd0051e170c,Siapa yang memprakarsai GONE theory dan apa sa...,"Teks menyatakan ""GONE theory yang diprakarsai ...",GONE theory yang diprakarsai oleh Jack Bologne...,Teks menyatakan GONE theory diprakarsai oleh J...,NaN,0.789916,0.638655,0.904712
2,9b46bc8bdc31522e,Menurut pendekatan lingkungan maupun kultural ...,"Teks menyatakan ""Dalam pendekatan lingkungan m...",Dalam pendekatan lingkungan maupun kultural ma...,Teks menyatakan bahwa solusi dalam pendekatan ...,NaN,0.804598,0.666667,0.907563
3,7e0baf29d0fa19e6,Sebutkan empat langkah pencegahan khusus korup...,"Teks menyatakan ""Heyneman (2004) menyatakan pa...","Berdasarkan konteks yang diberikan, dan inform...",Teks menyatakan bahwa Heyneman (2004) menyajik...,1.0,0.735294,0.647059,0.894974
4,8eaf6490b590c374,Apa yang dimaksud dengan GONE theory dalam kon...,"Teks menyatakan ""GONE theory yang diprakarsai ...","GONE theory, yang diprakarsai oleh Jack Bologn...",Teks menyatakan GONE theory menjelaskan empat ...,NaN,0.888889,0.793651,0.921785
...,...,...,...,...,...,...,...,...,...
1786,03460d992d3fd3a0,Apa sumber data yang digunakan untuk menghasil...,"Teks menyatakan ""Badan Pusat Statistik, Survei...","Berdasarkan konteks yang Anda berikan, sumber ...",Teks menyatakan sumber data adalah Badan Pusat...,NaN,0.873563,0.850575,0.896189
1787,3f25cc3863855ead,Apa yang diukur oleh indikator 'Persentase Pen...,"Teks menyatakan indikator tersebut adalah ""Per...",Indikator 'Persentase Penduduk yang Pernah Raw...,Teks menyatakan bahwa tabel ini mengukur perse...,NaN,0.672897,0.598131,0.904317
1788,94755286a0fca43d,Di bagian mana data mengenai akses rumah tangg...,"Teks menyatakan ""Lanjutan Tabel / Continued Ta...",Data mengenai akses rumah tangga terhadap fasi...,"Teks menyatakan ""Lanjutan Tabel / Continued Ta...",1.0,0.852941,0.823529,0.943995
1789,c97c2a058036c218,Di bagian mana informasi tentang sumber data u...,"Teks menyatakan ""Sumber/ Source : Badan Pusat ...",Informasi tentang sumber data untuk tabel samp...,Teks menyatakan bahwa sumber data untuk tabel ...,NaN,0.571429,0.571429,0.794784


In [16]:
## LLAMA-3.2 - BASE MODEL (SEBELUM FINE-TUNING)
 
df_llama_base = evaluate_condition(df_llama, pred_column="ans_pred_base")
df_llama_base

indobertscore 16/1791
indobertscore 32/1791
indobertscore 48/1791
indobertscore 64/1791
indobertscore 80/1791
indobertscore 96/1791
indobertscore 112/1791
indobertscore 128/1791
indobertscore 144/1791
indobertscore 160/1791
indobertscore 176/1791
indobertscore 192/1791
indobertscore 208/1791
indobertscore 224/1791
indobertscore 240/1791
indobertscore 256/1791
indobertscore 272/1791
indobertscore 288/1791
indobertscore 304/1791
indobertscore 320/1791
indobertscore 336/1791
indobertscore 352/1791
indobertscore 368/1791
indobertscore 384/1791
indobertscore 400/1791
indobertscore 416/1791
indobertscore 432/1791
indobertscore 448/1791
indobertscore 464/1791
indobertscore 480/1791
indobertscore 496/1791
indobertscore 512/1791
indobertscore 528/1791
indobertscore 544/1791
indobertscore 560/1791
indobertscore 576/1791
indobertscore 592/1791
indobertscore 608/1791
indobertscore 624/1791
indobertscore 640/1791
indobertscore 656/1791
indobertscore 672/1791
indobertscore 688/1791
indobertscore 704

,record_id,instruction,ans_ref,ans_pred_base,ans_pred_finetuned,em,f1,rouge_l,indobertscore
0,b05d48cbd7388514,Apa rekomendasi pertama dari KPK yang disusun ...,"Teks menyatakan ""Pertama, diperlukan pengemban...","Berdasarkan SPI Pendidikan 2020, salah satu re...","Teks menyatakan ""Pertama, diperlukan pengemban...",NaN,0.305085,0.305085,0.645569
1,6fd9fbd0051e170c,Siapa yang memprakarsai GONE theory dan apa sa...,"Teks menyatakan ""GONE theory yang diprakarsai ...","Menurut konteks yang diberikan, teori GONE yan...",Teks menyatakan bahwa GONE theory diprakarsai ...,NaN,0.666667,0.630631,0.858214
2,9b46bc8bdc31522e,Menurut pendekatan lingkungan maupun kultural ...,"Teks menyatakan ""Dalam pendekatan lingkungan m...",Menurut pendekatan lingkungan maupun kultural ...,Teks menyatakan bahwa dalam pendekatan lingkun...,NaN,0.375000,0.312500,0.633947
3,7e0baf29d0fa19e6,Sebutkan empat langkah pencegahan khusus korup...,"Teks menyatakan ""Heyneman (2004) menyatakan pa...","Menurut Heyneman (2004), empat langkah pencega...","Teks menyatakan ""Pertama adalah diperlukan ref...",1.0,0.569231,0.538462,0.832186
4,8eaf6490b590c374,Apa yang dimaksud dengan GONE theory dalam kon...,"Teks menyatakan ""GONE theory yang diprakarsai ...",Menurut teori GONE yang diprakarsai Jack Bolog...,Teks menyatakan bahwa GONE theory menjelaskan ...,NaN,0.296578,0.197719,0.602993
...,...,...,...,...,...,...,...,...,...
1786,03460d992d3fd3a0,Apa sumber data yang digunakan untuk menghasil...,"Teks menyatakan ""Badan Pusat Statistik, Survei...","Menurut Badan Pusat Statistik, Survei Sosial E...","Teks menyatakan ""Sumber/ Source : Badan Pusat ...",NaN,0.487179,0.435897,0.747502
1787,3f25cc3863855ead,Apa yang diukur oleh indikator 'Persentase Pen...,"Teks menyatakan indikator tersebut adalah ""Per...",Indikator tersebut mengukur persentase dari to...,"Teks menyatakan ""Persentase Penduduk yang Pern...",NaN,0.347826,0.289855,0.816099
1788,94755286a0fca43d,Di bagian mana data mengenai akses rumah tangg...,"Teks menyatakan ""Lanjutan Tabel / Continued Ta...",Data mengenai akses rumah tangga terhadap fasi...,"Teks menyatakan ""Lanjutan Tabel / Continued Ta...",1.0,0.536585,0.365854,0.816221
1789,c97c2a058036c218,Di bagian mana informasi tentang sumber data u...,"Teks menyatakan ""Sumber/ Source : Badan Pusat ...","Menurut konteks yang diberikan, informasi tent...","Teks menyatakan ""Sumber/ Source : Badan Pusat ...",NaN,0.312500,0.187500,0.557601


In [17]:
## LLAMA-3.2 - FINE-TUNED MODEL (SESUDAH FINE-TUNING)
 
df_llama_ft = evaluate_condition(df_llama, pred_column="ans_pred_finetuned")
df_llama_ft

indobertscore 16/1791
indobertscore 32/1791
indobertscore 48/1791
indobertscore 64/1791
indobertscore 80/1791
indobertscore 96/1791
indobertscore 112/1791
indobertscore 128/1791
indobertscore 144/1791
indobertscore 160/1791
indobertscore 176/1791
indobertscore 192/1791
indobertscore 208/1791
indobertscore 224/1791
indobertscore 240/1791
indobertscore 256/1791
indobertscore 272/1791
indobertscore 288/1791
indobertscore 304/1791
indobertscore 320/1791
indobertscore 336/1791
indobertscore 352/1791
indobertscore 368/1791
indobertscore 384/1791
indobertscore 400/1791
indobertscore 416/1791
indobertscore 432/1791
indobertscore 448/1791
indobertscore 464/1791
indobertscore 480/1791
indobertscore 496/1791
indobertscore 512/1791
indobertscore 528/1791
indobertscore 544/1791
indobertscore 560/1791
indobertscore 576/1791
indobertscore 592/1791
indobertscore 608/1791
indobertscore 624/1791
indobertscore 640/1791
indobertscore 656/1791
indobertscore 672/1791
indobertscore 688/1791
indobertscore 704

,record_id,instruction,ans_ref,ans_pred_base,ans_pred_finetuned,em,f1,rouge_l,indobertscore
0,b05d48cbd7388514,Apa rekomendasi pertama dari KPK yang disusun ...,"Teks menyatakan ""Pertama, diperlukan pengemban...","Berdasarkan SPI Pendidikan 2020, salah satu re...","Teks menyatakan ""Pertama, diperlukan pengemban...",NaN,0.972222,0.972222,0.984084
1,6fd9fbd0051e170c,Siapa yang memprakarsai GONE theory dan apa sa...,"Teks menyatakan ""GONE theory yang diprakarsai ...","Menurut konteks yang diberikan, teori GONE yan...",Teks menyatakan bahwa GONE theory diprakarsai ...,NaN,0.821429,0.732143,0.939397
2,9b46bc8bdc31522e,Menurut pendekatan lingkungan maupun kultural ...,"Teks menyatakan ""Dalam pendekatan lingkungan m...",Menurut pendekatan lingkungan maupun kultural ...,Teks menyatakan bahwa dalam pendekatan lingkun...,NaN,0.826087,0.717391,0.863519
3,7e0baf29d0fa19e6,Sebutkan empat langkah pencegahan khusus korup...,"Teks menyatakan ""Heyneman (2004) menyatakan pa...","Menurut Heyneman (2004), empat langkah pencega...","Teks menyatakan ""Pertama adalah diperlukan ref...",1.0,0.775956,0.710383,0.863165
4,8eaf6490b590c374,Apa yang dimaksud dengan GONE theory dalam kon...,"Teks menyatakan ""GONE theory yang diprakarsai ...",Menurut teori GONE yang diprakarsai Jack Bolog...,Teks menyatakan bahwa GONE theory menjelaskan ...,NaN,0.764228,0.666667,0.874965
...,...,...,...,...,...,...,...,...,...
1786,03460d992d3fd3a0,Apa sumber data yang digunakan untuk menghasil...,"Teks menyatakan ""Badan Pusat Statistik, Survei...","Menurut Badan Pusat Statistik, Survei Sosial E...","Teks menyatakan ""Sumber/ Source : Badan Pusat ...",NaN,0.867470,0.771084,0.893809
1787,3f25cc3863855ead,Apa yang diukur oleh indikator 'Persentase Pen...,"Teks menyatakan indikator tersebut adalah ""Per...",Indikator tersebut mengukur persentase dari to...,"Teks menyatakan ""Persentase Penduduk yang Pern...",NaN,0.732143,0.714286,0.939939
1788,94755286a0fca43d,Di bagian mana data mengenai akses rumah tangg...,"Teks menyatakan ""Lanjutan Tabel / Continued Ta...",Data mengenai akses rumah tangga terhadap fasi...,"Teks menyatakan ""Lanjutan Tabel / Continued Ta...",1.0,0.794521,0.739726,0.928251
1789,c97c2a058036c218,Di bagian mana informasi tentang sumber data u...,"Teks menyatakan ""Sumber/ Source : Badan Pusat ...","Menurut konteks yang diberikan, informasi tent...","Teks menyatakan ""Sumber/ Source : Badan Pusat ...",NaN,0.739130,0.717391,0.901721


## 4. Ringkasan

In [18]:
## RINGKASAN PERBANDINGAN
 
summary = pd.DataFrame({
    "model": ["gemma2_base", "gemma2_finetuned", "llama3.2_base", "llama3.2_finetuned"],
    "em": [df_gemma_base["em"].mean(), df_gemma_ft["em"].mean(), df_llama_base["em"].mean(), df_llama_ft["em"].mean()],
    "f1": [df_gemma_base["f1"].mean(), df_gemma_ft["f1"].mean(), df_llama_base["f1"].mean(), df_llama_ft["f1"].mean()],
    "rouge_l": [df_gemma_base["rouge_l"].mean(), df_gemma_ft["rouge_l"].mean(), df_llama_base["rouge_l"].mean(), df_llama_ft["rouge_l"].mean()],
    "indobertscore": [df_gemma_base["indobertscore"].mean(), df_gemma_ft["indobertscore"].mean(), df_llama_base["indobertscore"].mean(), df_llama_ft["indobertscore"].mean()],
})
summary

,model,em,f1,rouge_l,indobertscore
0,gemma2_base,0.593248,0.459312,0.373543,0.774076
1,gemma2_finetuned,0.695338,0.717424,0.642917,0.888417
2,llama3.2_base,0.596463,0.451857,0.361699,0.768241
3,llama3.2_finetuned,0.733119,0.725324,0.651080,0.890660


## 5. Simpan

In [19]:
api = HfApi(token=HF_TOKEN)
 
detail_outputs = {
    "gemma2_base": df_gemma_base,
    "gemma2_finetuned": df_gemma_ft,
    "llama3.2_base": df_llama_base,
    "llama3.2_finetuned": df_llama_ft,
}
 
for name, df_result in detail_outputs.items():
    path = f"{OUTPUT_DIR}/{name}_scored_gguf.csv"
    df_result.to_csv(path, index=False)
    api.upload_file(path_or_fileobj=path, path_in_repo=f"{name}_scored_gguf.csv", repo_id=HF_REPO, repo_type="dataset")
    print(f"Ter-push: {name}_scored_gguf.csv")
 
summary_path = f"{OUTPUT_DIR}/summary_metrics_gguf.csv"
summary.to_csv(summary_path, index=False)
api.upload_file(path_or_fileobj=summary_path, path_in_repo="summary_metrics_gguf.csv", repo_id=HF_REPO, repo_type="dataset")
print("Ter-push: summary_metrics_gguf.csv")

Ter-push: gemma2_base_scored_gguf.csv
Ter-push: gemma2_finetuned_scored_gguf.csv
Ter-push: llama3.2_base_scored_gguf.csv
Ter-push: llama3.2_finetuned_scored_gguf.csv
Ter-push: summary_metrics_gguf.csv
